# Factualis: final evaluation and website

This notebook does not train anything. It picks up the model trained in `1_full_pipeline.ipynb`, compares it with the untrained model on the 94 test questions, and then starts the website with whichever of the two did better.

You need the same `factualis_colab.zip` and the run folder `MyDrive/Factualis/runs/run1/` from the first notebook. Switch the runtime to a T4 and use Runtime > Run all. The last cell prints the link to the website.

The evaluation takes about 25 minutes the first time and is skipped afterwards because the results are saved on Drive. After that, starting the website takes 10 to 15 minutes, most of it installing SGLang.

### Settings

`SERVE_MODEL` can be `"base"`, `"trained"` or `"auto"`. With `"auto"` the notebook serves whichever model gets more answers both usable and correct in the evaluation below; on a tie it keeps the base model, which refuses out-of-policy questions more reliably.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Factualis"
RUN_NAME = "run1"
BASE = "Qwen/Qwen2.5-3B-Instruct"
SGLANG_SPEC = "sglang[all]"
SERVE_MODEL = "auto"
RUN_EVALUATION = True

### Mount Drive and unpack the code

In [ ]:
import os, json, re, shutil, subprocess, time, zipfile, pathlib, urllib.request
from google.colab import drive
drive.mount("/content/drive")

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
assert "T4" in gpu, "Switch the runtime to a T4 GPU (Runtime > Change runtime type)."

PROJ = "/content/Factualis"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"
CKPT = "/content/checkpoints"
DRIVE_CKPT = f"{RUN_DIR}/checkpoints"
for d in (RUN_DIR, f"{RUN_DIR}/output", DRIVE_CKPT, CKPT):
    os.makedirs(d, exist_ok=True)

zip_path = f"{DRIVE_DIR}/factualis_colab.zip"
assert os.path.exists(zip_path), f"Upload factualis_colab.zip to {DRIVE_DIR} first."
shutil.rmtree(PROJ, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(PROJ)
shutil.rmtree(f"{PROJ}/output", ignore_errors=True)
os.symlink(f"{RUN_DIR}/output", f"{PROJ}/output")
os.chdir(PROJ)
print("code unpacked to", PROJ, "| results go to", RUN_DIR)

MODEL_DRIVE = f"{DRIVE_CKPT}/qwen3b-fincomp-dpo-merged"

### Install what the website needs

Only two environments this time: SGLang for serving, and a small one for the web app and AuditKIT. If the trained model has to be rebuilt from its LoRA adapters, a third environment with the merge tools is added.

In [ ]:
NEED_MERGE = SERVE_MODEL == "trained" and not os.path.exists(f"{MODEL_DRIVE}/config.json")
ENVS = {"serve": f'"{SGLANG_SPEC}" openai ninja', "app": "fastapi uvicorn openai pydantic rank_bm25 auditkit"}
if NEED_MERGE:
    ENVS["merge"] = "torch transformers peft accelerate safetensors"

subprocess.run("pip -q install uv", shell=True, check=True)
procs = {}
for name, pkgs in ENVS.items():
    root = f"/content/envs/{name}"
    if os.path.exists(f"{root}/.ok"):
        continue
    cmd = f"uv venv -q --python 3.11 {root} && uv pip install -q --python {root}/bin/python {pkgs} && touch {root}/.ok"
    procs[name] = subprocess.Popen(cmd, shell=True, stdout=open(f"/content/install_{name}.log", "w"), stderr=subprocess.STDOUT)
for name, p in procs.items():
    rc = p.wait()
    print(f"{name}: {'ok' if rc == 0 else 'failed, see /content/install_' + name + '.log'}")
    if rc:
        print(open(f"/content/install_{name}.log").read()[-3000:])

PY = {k: f"/content/envs/{k}/bin/python" for k in ENVS}
BIN = {k: f"/content/envs/{k}/bin" for k in ENVS}
for k in ENVS:
    assert os.path.exists(f"/content/envs/{k}/.ok"), f"environment {k} did not install"
print("environments ready")

### Helper functions

A few small functions the rest of the notebook uses:

- `sh` runs a command inside the unpacked project and streams its output.
- `start_sglang` / `stop_sglang` start and stop the model server. Only one model fits on the T4 at a time, so every step that needs a different model stops the old server first. The server is started in its own process session, otherwise pressing Colab's stop button on any cell would also kill it.
- `open_website` starts the FastAPI app and a free Cloudflare tunnel, then prints the public link.

The `pkill` patterns are written as `web[.]app` and `sglang[.]launch_server` on purpose. A plain `web.app` also matches Colab's own `/datalab/web/app.js` process, and killing that disconnects the runtime.

In [ ]:
def sh(cmd, env=None):
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, cwd=PROJ, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed: {cmd}")


def done(path):
    return os.path.exists(path)


def gpu_used_mb():
    out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                         shell=True, capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])


def stop_sglang():
    subprocess.run("pkill -f 'sglang[.]launch_server'", shell=True)
    for i in range(60):
        if gpu_used_mb() < 1000:
            break
        if i == 20:
            subprocess.run("pkill -9 -f 'sglang[.]launch_server'", shell=True)
        time.sleep(2)
    print(f"SGLang stopped, GPU memory in use: {gpu_used_mb()} MiB")


def start_sglang(model, port=30000, mem=0.80, extra="", timeout=1200):
    stop_sglang()
    flags = ("--dtype half --attention-backend triton --sampling-backend pytorch --grammar-backend xgrammar "
             "--context-length 8192 --served-model-name default --host 0.0.0.0 --enable-metrics")
    if "AWQ" in model:
        flags += " --quantization awq"
    log = f"/content/sglang_{port}.log"
    cmd = f"{PY['serve']} -m sglang.launch_server --model-path {model} --port {port} --mem-fraction-static {mem} {flags} {extra}"
    print(f"$ {cmd}")
    env = {**os.environ, "PATH": f"{BIN['serve']}:/usr/local/cuda/bin:{os.environ['PATH']}", "CUDA_HOME": "/usr/local/cuda"}
    p = subprocess.Popen(cmd, shell=True, cwd=PROJ, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(f"http://localhost:{port}/health", timeout=30)
            print(f"SGLang ready after {time.time() - t0:.0f}s ({model})")
            return
        except Exception:
            if p.poll() is not None:
                print(open(log).read()[-3000:])
                raise RuntimeError(f"SGLang exited, full log in {log}")
            time.sleep(5)
    raise TimeoutError(f"SGLang not ready after {timeout}s, see {log}")


def open_website(extra_args=""):
    subprocess.run("pkill -f 'm web[.]app --port'; pkill -f 'cloudflared tunnel'", shell=True)
    time.sleep(1)
    subprocess.Popen(f"{PY['app']} -u -m web.app --port 8000 --aligned-port 30000 {extra_args}", shell=True, cwd=PROJ,
                     stdout=open("/content/web.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://localhost:8000/api/examples", timeout=3)
            break
        except Exception:
            time.sleep(2)
    else:
        print(open("/content/web.log").read()[-2000:])
        return
    if not done("/content/cloudflared"):
        sh("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared")
    subprocess.Popen("/content/cloudflared tunnel --no-autoupdate --url http://localhost:8000", shell=True,
                     stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[\w-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            print("\nFactualis is live at:", m.group(0))
            return
    print(open("/content/tunnel.log").read()[-1500:])

## Evaluate both models

Both models answer the 94 test questions with the answer format enforced, and AuditKIT grades and compares the two runs question by question. The results are written to `output/eval_v2` on Drive; the folder and run names are the ones our existing results already use, so a second run of this notebook skips straight past this step.

In [ ]:
RESULTS = "output/eval_v2"
BASE_RUN, TRAINED_RUN = "B2_base_v2", "D2_aligned_v2"
HOLDOUT_ARGS = "--holdout-doc sebi_pit_regulations --holdout-doc fb_personal_dealing_policy"

if RUN_EVALUATION:
    runs = [(BASE_RUN, BASE)]
    if os.path.exists(f"{MODEL_DRIVE}/config.json"):
        runs.append((TRAINED_RUN, MODEL_DRIVE))
    for name, model in runs:
        if not os.path.exists(f"{RESULTS}/{name}.json"):
            start_sglang(model)
            sh(f"{PY['app']} -m pipeline.evaluate --name {name} --results-dir {RESULTS} {HOLDOUT_ARGS}")
    sh(f"{PY['app']} -m pipeline.evaluate --report --results-dir {RESULTS}")
    sh(f"{PY['app']} -m pipeline.compare_with_auditkit --results-dir {RESULTS} --baseline {BASE_RUN} --candidates {TRAINED_RUN}")


def usable_correct(run):
    path = f"{RESULTS}/{run}.json"
    return json.load(open(path)).get("usable_correct_rate", 0) if os.path.exists(path) else -1


if SERVE_MODEL == "auto":
    SERVE_MODEL = "trained" if usable_correct(TRAINED_RUN) > usable_correct(BASE_RUN) else "base"
    print(f"usable and correct: base {usable_correct(BASE_RUN):.3f}, trained {usable_correct(TRAINED_RUN):.3f}, serving {SERVE_MODEL}")

## Get the model ready

If the trained model is going to be served and the merged copy is not on Drive yet, it is rebuilt once from the SFT and DPO adapters and saved, so later runs can skip this.

In [ ]:
NEED_MERGE = SERVE_MODEL == "trained" and not os.path.exists(f"{MODEL_DRIVE}/config.json")
if NEED_MERGE:
    sft_adapter, dpo_adapter = f"{DRIVE_CKPT}/qwen3b-fincomp-sft", f"{DRIVE_CKPT}/qwen3b-fincomp-dpo"
    assert os.path.exists(f"{dpo_adapter}/adapter_config.json"), f"no trained adapters found in {DRIVE_CKPT}"
    sft_merged, dpo_merged = f"{CKPT}/qwen3b-fincomp-sft-merged", f"{CKPT}/qwen3b-fincomp-dpo-merged"
    stop_sglang()
    sh(f"{PY['merge']} -m pipeline.train merge --adapter {sft_adapter} --base {BASE} --out {sft_merged}")
    sh(f"{PY['merge']} -m pipeline.train merge --adapter {dpo_adapter} --base {sft_merged} --out {dpo_merged}")
    shutil.copytree(dpo_merged, MODEL_DRIVE, dirs_exist_ok=True)
    MODEL = dpo_merged
else:
    MODEL = MODEL_DRIVE if SERVE_MODEL == "trained" else BASE
print("serving:", MODEL)

## Start the website

When the trained model is served, the page can also show the untrained model's answers to the example questions under "Before training". Those are computed once and cached on Drive. When the base model is served, that switch is hidden.

In [ ]:
examples_cache = "output/demo_base_cache_v2.json"
compare = SERVE_MODEL == "trained"
if compare and not os.path.exists(examples_cache):
    start_sglang(BASE)
    sh(f"{PY['app']} -m web.app --precompute-examples {examples_cache}")

start_sglang(MODEL)
open_website(f"--base-cache {examples_cache}" if compare else "--no-compare")

If the page says the model is not available, run the cell above again. The logs are in `/content/sglang_30000.log` and `/content/web.log` if you need to see what went wrong.